# 02 · Model Evaluation — All Four Split Strategies
Loads the saved checkpoint and computes the paper-ready headline table.

| Split | Purpose |
|---|---|
| **Group k-fold (HEADLINE)** | True generalisation to unseen experiment cells |
| Random split | Baseline — shows inflated F1 due to leakage |
| Time-aware split | Temporal ordering — last 15% by cycle ID |
| Material OOD (PP → ABS) | Domain shift: train PP, test ABS recyclate |

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

NB_DIR  = Path.cwd()
FS_DIR  = NB_DIR.parent
sys.path.insert(0, str(FS_DIR))

import config as C
from data.loaders    import load_parquet, build_cv_index, join_cv_to_parquet, load_tabular_vector
from data.dataset    import MultiModalDataset, collate_fn, get_tabular_cols, get_dxp_cols
from models.fusion   import InjectionMouldingFusionModel
from training.splits import (group_kfold_splits, random_split,
                              time_aware_split, material_ood_split,
                              build_experiment_label_map)
from training.losses  import TotalLoss
from training.metrics import (compute_metrics, find_best_thresholds,
                               group_aware_evaluation, build_headline_table)
from training.trainer import run_epoch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# ── Load data ────────────────────────────────────────────────
df     = load_parquet(C.PARQUET_PATH)
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)

tabular_cols  = get_tabular_cols(df)
dxp_cols      = get_dxp_cols(df)
exp_label_map = build_experiment_label_map(df)

saved_thrs = None
thr_path   = C.MODELS_DIR / "best_thresholds.npy"
if thr_path.exists():
    saved_thrs = np.load(thr_path)
    print("Loaded thresholds from:", thr_path)

def _build_loader(eval_df, ref_df, batch=C.BATCH_SIZE):
    scaler = StandardScaler()
    scaler.fit(np.array([load_tabular_vector(r, tabular_cols) for _, r in ref_df.iterrows()]))
    tab_med = {c: float(ref_df[c].median()) for c in tabular_cols if c in ref_df.columns}
    roi_med = {c: float(ref_df[c].median()) for c in C.THERMAL_ROI_COLS if c in ref_df.columns}
    ds = MultiModalDataset(eval_df, tabular_cols, dxp_cols, train=False,
                           tabular_medians=tab_med, roi_medians=roi_med, tabular_scaler=scaler)
    return DataLoader(ds, batch_size=batch, shuffle=False, collate_fn=collate_fn, num_workers=0)

def _load_model():
    m = InjectionMouldingFusionModel(
        tabular_in_dim=len(tabular_cols),
        n_dxp_channels=len(dxp_cols),
        n_experiments=len(exp_label_map),
    ).to(DEVICE)
    ckpt = C.MODELS_DIR / "fusion_final.pt"
    if not ckpt.exists():
        raise FileNotFoundError(f"No checkpoint at {ckpt}. Run 01_train_colab.ipynb first.")
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    return m

model = _load_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} params")

## 1 · Group K-Fold (HEADLINE)

In [ ]:
all_probs, all_targets, all_groups = [], [], []
for fold_i, (train_df, val_df) in enumerate(group_kfold_splits(df)):
    ld = _build_loader(val_df, train_df)
    _, vp, vt, vg = run_epoch(model, ld, TotalLoss(), None, DEVICE, exp_label_map, train=False)
    all_probs.append(vp); all_targets.append(vt); all_groups.append(vg)
    print(f"Fold {fold_i+1}: {len(val_df)} val parts")

all_p = np.concatenate(all_probs)
all_t = np.concatenate(all_targets)
all_g = np.concatenate(all_groups)
thrs_gkf   = find_best_thresholds(all_p, all_t) if saved_thrs is None else saved_thrs
gkf_metrics = compute_metrics(all_p, all_t, thresholds=thrs_gkf)
gkf_break   = group_aware_evaluation(all_p, all_t, all_g, thresholds=thrs_gkf)
print(f"\nGroup K-Fold Macro-F1 : {gkf_metrics['macro_f1']:.4f}")
print(f"Per-group mean±std    : {gkf_break['mean_group_f1']:.4f} ± {gkf_break['std_group_f1']:.4f}")

## 2 · Random Split Baseline

In [ ]:
tr, va, te = random_split(df)
ld_rand = _build_loader(te, tr)
_, rp, rt, rg = run_epoch(model, ld_rand, TotalLoss(), None, DEVICE, exp_label_map, train=False)
thrs_rand  = find_best_thresholds(rp, rt)
rand_metrics = compute_metrics(rp, rt, thresholds=thrs_rand)
print(f"Random Split Macro-F1 : {rand_metrics['macro_f1']:.4f}  (inflated — leakage!)")

## 3 · Time-Aware Split

In [ ]:
tr_t, va_t, te_t = time_aware_split(df)
ld_time = _build_loader(te_t, tr_t)
_, tp, tt, tg = run_epoch(model, ld_time, TotalLoss(), None, DEVICE, exp_label_map, train=False)
thrs_time    = find_best_thresholds(tp, tt)
time_metrics = compute_metrics(tp, tt, thresholds=thrs_time)
print(f"Time-Aware Macro-F1   : {time_metrics['macro_f1']:.4f}")

## 4 · Material OOD  (PP → ABS)

In [ ]:
if C.MATERIAL_COL not in df.columns:
    print("MATERIAL_COL not found — skipping OOD evaluation")
    ood_metrics = {"macro_f1": float("nan"), "micro_f1": float("nan"),
                   "mean_roc_auc": float("nan"), "mean_pr_auc": float("nan"),
                   "per_label_f1": {}}
else:
    ood_tr, ood_te = material_ood_split(df)
    if len(ood_te) == 0:
        print("No ABS parts in dataset — OOD eval skipped")
        ood_metrics = {"macro_f1": float("nan"), "micro_f1": float("nan"),
                       "mean_roc_auc": float("nan"), "mean_pr_auc": float("nan"),
                       "per_label_f1": {}}
    else:
        ld_ood = _build_loader(ood_te, ood_tr)
        _, op, ot, og = run_epoch(model, ld_ood, TotalLoss(), None, DEVICE, exp_label_map, train=False)
        thrs_ood   = find_best_thresholds(op, ot)
        ood_metrics = compute_metrics(op, ot, thresholds=thrs_ood)
        print(f"OOD (PP→ABS) Macro-F1 : {ood_metrics['macro_f1']:.4f}")

## 5 · Paper Headline Table

In [ ]:
results_map = {
    "Group k-fold (HEADLINE)" : gkf_metrics,
    "Random split (baseline)" : rand_metrics,
    "Time-aware split"        : time_metrics,
    "Material OOD (PP→ABS)"  : ood_metrics,
}
headline = build_headline_table(results_map)
print("=" * 60)
print("HEADLINE EVALUATION TABLE")
print("=" * 60)
print(headline.to_string(index=False))
print("=" * 60)

headline.to_csv(C.ARTIFACTS_DIR / "evaluation_headline.csv", index=False)
print("Saved to:", C.ARTIFACTS_DIR / "evaluation_headline.csv")

## 6 · Confusion Matrix & Per-Label F1

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt, numpy as np

# Confusion matrix — LBL_NOK — group k-fold
cm = np.array(gkf_metrics["nok_confusion"])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["OK","NOK"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — LBL_NOK (Group k-fold)")

# Per-label F1 comparison
labels = C.LABEL_COLS
x = np.arange(len(labels))
w = 0.22
for ki, (sname, smet) in enumerate(results_map.items()):
    pf1 = [smet["per_label_f1"].get(l, 0) for l in labels]
    axes[1].bar(x + ki*w, pf1, w, label=sname, alpha=0.85)

axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels([l.replace("LBL_","") for l in labels], rotation=30, ha="right")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("F1 Score")
axes[1].set_title("Per-Label F1 — All Split Strategies")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(C.ARTIFACTS_DIR / "evaluation_figure.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Embedding Space — t-SNE

In [ ]:
from sklearn.manifold import TSNE
import torch, numpy as np

# Collect fused embeddings + labels
embs, ys, grps = [], [], []
ld_all = _build_loader(df.head(200), df.head(200))  # quick subset
model.eval()
with torch.no_grad():
    for batch in ld_all:
        b = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = model(b)
        embs.append(out["fused_emb"].cpu().numpy())
        ys.append(batch["labels"][:, C.LABEL_COLS.index(C.PRIMARY_TARGET)].numpy())
        grps.extend(batch["group"])

embs_arr = np.concatenate(embs)
ys_arr   = np.concatenate(ys)

print("Running t-SNE on fused embeddings...")
tsne  = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embs2 = tsne.fit_transform(embs_arr)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(embs2[:, 0], embs2[:, 1], c=ys_arr,
                     cmap="RdYlGn_r", alpha=0.7, s=40, edgecolors="white", linewidths=0.3)
plt.colorbar(scatter, ax=ax, label="LBL_NOK")
ax.set_title("t-SNE of Fused Embeddings — coloured by LBL_NOK")
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
plt.tight_layout(); plt.show()
print("Well-separated clusters → model has learned discriminative representations")